# Thesis Figure Generation

This notebook generates the publication-ready figures used in the thesis results chapter and appendix. It loads existing training statistics, exported model JSON files, and saved model-output WAV files. It does not retrain models or run inference.

In [1]:
from pathlib import Path
import json
import math
import os

# Keep Matplotlib cache writes inside the repository/sandbox instead of ~/.matplotlib.
os.environ.setdefault("MPLCONFIGDIR", str(Path.cwd() / ".matplotlib-cache"))

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy.io import wavfile
from scipy import signal
from IPython.display import display, Markdown

Matplotlib is building the font cache; this may take a moment.


## Paths and Figure Style

All figures are exported to `thesis/images/figures/` so they can be referenced directly from the LaTeX thesis sources.

In [2]:
def find_model_root() -> Path:
    cwd = Path.cwd().resolve()
    if (cwd / "Data").exists() and (cwd / "Results").exists():
        return cwd
    candidate = cwd / "model"
    if (candidate / "Data").exists() and (candidate / "Results").exists():
        return candidate.resolve()
    raise RuntimeError("Run this notebook from the repository root or the model/ directory.")

ROOT = find_model_root()
REPO_ROOT = ROOT.parent
DATA_DIR = ROOT / "Data"
RESULTS_DIR = ROOT / "Results"
STATS_DIR = ROOT / "notebook_stats"
FIGURE_DIR = REPO_ROOT / "thesis" / "images" / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "ds1"
RUNS = {
    "LSTM": {
        "stats_path": STATS_DIR / "ds1_LSTM_stats.json",
        "result_dir": RESULTS_DIR / "ds1-LSTM",
        "color": "#2f5d8c",
        "linestyle": "-",
        "marker": "o",
    },
    "GRU": {
        "stats_path": STATS_DIR / "ds1_GRU_stats.json",
        "result_dir": RESULTS_DIR / "ds1-GRU",
        "color": "#6f6f6f",
        "linestyle": "--",
        "marker": "s",
    },
}

mpl.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.04,
    "font.family": "serif",
    "font.serif": ["DejaVu Serif", "Times New Roman", "Times"],
    "font.size": 9,
    "axes.labelsize": 9,
    "axes.titlesize": 9,
    "axes.linewidth": 0.7,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
    "lines.linewidth": 1.35,
    "lines.markersize": 3.2,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

PANEL_KW = dict(fontweight="bold", va="top", ha="left")

def save_figure(fig, filename: str, dpi: int = 300):
    path = FIGURE_DIR / filename
    fig.savefig(path, dpi=dpi)
    plt.close(fig)
    return path

def panel_label(ax, label: str, x: float = 0.01, y: float = 0.97):
    ax.text(x, y, label, transform=ax.transAxes, **PANEL_KW)

def style_axis(ax, grid=True):
    if grid:
        ax.grid(True, which="major", color="#d9d9d9", linewidth=0.45, alpha=0.9)
        ax.grid(True, which="minor", color="#eeeeee", linewidth=0.35, alpha=0.7)
    ax.tick_params(direction="out", width=0.7, length=3)
    return ax

## Loading and Metric Helpers

In [3]:
def require_file(path: Path) -> Path:
    if not path.exists():
        raise FileNotFoundError(f"Missing required artifact: {path}")
    return path


def load_json(path: Path):
    with require_file(path).open("r", encoding="utf-8") as f:
        return json.load(f)


def count_scalars(value) -> int:
    if isinstance(value, dict):
        return sum(count_scalars(v) for v in value.values())
    if isinstance(value, list):
        if not value:
            return 0
        return sum(count_scalars(v) for v in value)
    if isinstance(value, (int, float)):
        return 1
    return 0


def read_wav_float(path: Path):
    fs, data = wavfile.read(require_file(path))
    data = np.asarray(data)
    if data.ndim > 1:
        data = data[:, 0]
    if np.issubdtype(data.dtype, np.integer):
        info = np.iinfo(data.dtype)
        scale = max(abs(info.min), info.max)
        data = data.astype(np.float32) / scale
    else:
        data = data.astype(np.float32)
    return fs, data


def align_audio(*arrays):
    n = min(len(a) for a in arrays)
    return tuple(np.asarray(a[:n], dtype=np.float32) for a in arrays)


def objective_metrics(target, output):
    target, output = align_audio(target, output)
    error = output - target
    mse = float(np.mean(error ** 2))
    target_energy = float(np.mean(target ** 2))
    esr = mse / target_energy if target_energy > 0 else math.nan
    rmse = math.sqrt(mse)
    mae = float(np.mean(np.abs(error)))
    snr = 10 * math.log10(target_energy / mse) if mse > 0 and target_energy > 0 else math.inf
    corr = float(np.corrcoef(target, output)[0, 1]) if len(target) > 1 else math.nan
    return {"MSE": mse, "RMSE": rmse, "MAE": mae, "ESR": esr, "SNR (dB)": snr, "Correlation": corr}


def db(value, floor=1e-12):
    return 20 * np.log10(np.maximum(np.asarray(value), floor))


def excerpt_signal(x, fs, start_s, duration_s):
    start = max(0, int(round(start_s * fs)))
    n = int(round(duration_s * fs))
    end = min(len(x), start + n)
    return np.arange(end - start) / fs, x[start:end]


def select_excerpts(target, fs, duration_s=1.0, count=4, min_separation_s=1.0):
    short_win = int(0.08 * fs)
    hop = int(0.02 * fs)
    starts = np.arange(0, len(target) - short_win, hop)
    rms = np.array([np.sqrt(np.mean(target[s:s + short_win] ** 2)) for s in starts])
    transient = np.abs(np.diff(rms, prepend=rms[0]))
    peak = np.array([np.max(np.abs(target[s:s + short_win])) for s in starts])
    score = (rms / max(rms.max(), 1e-12)) + 0.8 * (transient / max(transient.max(), 1e-12)) + 0.3 * (peak / max(peak.max(), 1e-12))
    valid = (starts / fs > 1.0) & ((starts / fs + duration_s) < (len(target) / fs - 1.0))
    ranked = starts[valid][np.argsort(score[valid])[::-1]] / fs
    chosen = []
    for start_s in ranked:
        if all(abs(start_s - existing) >= min_separation_s for existing in chosen):
            chosen.append(float(start_s))
        if len(chosen) >= count:
            break
    return chosen


def spectrogram_db(x, fs, start_s, duration_s, nperseg=2048, noverlap=1536):
    _, segment = excerpt_signal(x, fs, start_s, duration_s)
    freqs, times, spec = signal.spectrogram(segment, fs=fs, window="hann", nperseg=nperseg, noverlap=noverlap, mode="magnitude")
    return freqs, times + start_s, db(spec)


def spectral_band_errors(target, output, fs, bands):
    target, output = align_audio(target, output)
    freqs, times, target_spec = signal.spectrogram(target, fs=fs, window="hann", nperseg=4096, noverlap=2048, mode="magnitude")
    _, _, output_spec = signal.spectrogram(output, fs=fs, window="hann", nperseg=4096, noverlap=2048, mode="magnitude")
    abs_error_db = np.abs(db(output_spec) - db(target_spec))
    rows = []
    for label, lo, hi in bands:
        mask = (freqs >= lo) & (freqs < hi)
        rows.append({"Band": label, "Mean Abs Spectral Error (dB)": float(np.mean(abs_error_db[mask]))})
    return rows

## Load Artifacts

In [4]:
stats = {name: load_json(run["stats_path"]) for name, run in RUNS.items()}

fs_target, test_target = read_wav_float(DATA_DIR / "test" / f"{DEVICE}-target.wav")
fs_input, test_input = read_wav_float(DATA_DIR / "test" / f"{DEVICE}-input.wav")
assert fs_target == fs_input, "Test input and target sample rates differ."

outputs = {}
for name, run in RUNS.items():
    result_dir = run["result_dir"]
    fs_best, best = read_wav_float(result_dir / "test_out_bestv.wav")
    fs_final, final = read_wav_float(result_dir / "test_out_final.wav")
    assert fs_best == fs_target and fs_final == fs_target, f"{name} output sample rate mismatch."
    outputs[name] = {"Best validation": best, "Final": final}

model_rows = []
for name, run in RUNS.items():
    model = load_json(run["result_dir"] / "model_best.json")
    model_rows.append({"Model": name, "Parameters": count_scalars(model["state_dict"])})
df_models = pd.DataFrame(model_rows)

metric_rows = []
for name, checkpoints in outputs.items():
    for checkpoint, output in checkpoints.items():
        row = {"Model": name, "Checkpoint": checkpoint}
        row.update(objective_metrics(test_target, output))
        metric_rows.append(row)
df_metrics = pd.DataFrame(metric_rows)

EXCERPTS = select_excerpts(test_target, fs_target, duration_s=1.0, count=4, min_separation_s=1.0)
MAIN_START = EXCERPTS[0]

summary = pd.DataFrame([
    {
        "Model": name,
        "Epochs": s["total_epochs_run"],
        "Best epoch": s["best_epoch"],
        "Best validation loss": s["best_val_loss"],
        "Best-validation ESR": df_metrics[(df_metrics["Model"] == name) & (df_metrics["Checkpoint"] == "Best validation")]["ESR"].iloc[0],
        "Parameters": df_models[df_models["Model"] == name]["Parameters"].iloc[0],
    }
    for name, s in stats.items()
])

display(summary.style.format({"Best validation loss": "{:.6f}", "Best-validation ESR": "{:.6f}"}).hide(axis="index"))
display(Markdown(f"Selected representative excerpt starts: {', '.join(f'{x:.2f} s' for x in EXCERPTS)}"))

Model,Epochs,Best epoch,Best validation loss,Best-validation ESR,Parameters
LSTM,336,284,0.023025,0.021411,2617
GRU,138,86,0.037394,0.035938,1969


Selected representative excerpt starts: 22.96 s, 16.20 s, 13.92 s, 14.92 s

## Figure 4.1 — Combined Training and Validation Loss Curves

In [5]:
fig, axes = plt.subplots(1, 2, figsize=(7.2, 2.8), sharex=False)

for name, run in RUNS.items():
    s = stats[name]
    epochs = np.arange(1, len(s["train_losses"]) + 1)
    axes[0].plot(epochs, s["train_losses"], color=run["color"], linestyle=run["linestyle"], label=name)
    axes[0].axvline(s["best_epoch"], color=run["color"], linestyle=":", linewidth=1.0, alpha=0.9)

    val_epochs = [ep for ep, _ in s["val_losses"]]
    val_values = [loss for _, loss in s["val_losses"]]
    axes[1].plot(val_epochs, val_values, color=run["color"], linestyle=run["linestyle"], marker=run["marker"], markevery=max(len(val_epochs)//12, 1), label=name)
    axes[1].axvline(s["best_epoch"], color=run["color"], linestyle=":", linewidth=1.0, alpha=0.9)

for ax, label, subtitle in zip(axes, ["(a)", "(b)"], ["Training loss", "Validation loss"]):
    style_axis(ax)
    panel_label(ax, label)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.text(0.12, 0.97, subtitle, transform=ax.transAxes, va="top", ha="left")
    ax.legend(frameon=False, loc="upper right")

path = save_figure(fig, "fig_4_1_loss_curves.pdf")
path

PosixPath('/Users/sathira/dev/boss-ds1/ds1-neural-emulation/thesis/images/figures/fig_4_1_loss_curves.pdf')

## Figure 4.2 — Learning-Rate Schedule

In [6]:
fig, ax = plt.subplots(figsize=(5.0, 2.8))
for name, run in RUNS.items():
    lr = stats[name]["learning_rates"]
    epochs = np.arange(1, len(lr) + 1)
    ax.step(epochs, lr, where="post", color=run["color"], linestyle=run["linestyle"], label=name)
style_axis(ax)
ax.set_yscale("log")
ax.set_xlabel("Epoch")
ax.set_ylabel("Learning rate")
ax.legend(frameon=False, loc="upper right")
path = save_figure(fig, "fig_4_2_learning_rate_schedule.pdf")
path

PosixPath('/Users/sathira/dev/boss-ds1/ds1-neural-emulation/thesis/images/figures/fig_4_2_learning_rate_schedule.pdf')

## Figure 4.3 — Representative Waveform Comparison

In [7]:
wave_duration = 0.12
series = [("Target", test_target), ("LSTM", outputs["LSTM"]["Best validation"]), ("GRU", outputs["GRU"]["Best validation"])]
segments = []
for label, x in series:
    t_rel, seg = excerpt_signal(x, fs_target, MAIN_START, wave_duration)
    segments.append((label, t_rel * 1000, seg))
ymax = max(np.max(np.abs(seg)) for _, _, seg in segments) * 1.05

fig, axes = plt.subplots(3, 1, figsize=(7.2, 4.2), sharex=True, sharey=True)
for idx, (ax, (label, t_ms, seg)) in enumerate(zip(axes, segments)):
    color = "#333333" if label == "Target" else RUNS[label]["color"]
    linestyle = "-" if label == "Target" else RUNS[label]["linestyle"]
    ax.plot(t_ms, seg, color=color, linestyle=linestyle, linewidth=0.9)
    style_axis(ax)
    panel_label(ax, f"({chr(ord('a') + idx)})")
    ax.text(0.10, 0.92, label, transform=ax.transAxes, va="top", ha="left")
    ax.set_ylabel("Amplitude")
    ax.set_ylim(-ymax, ymax)
axes[-1].set_xlabel("Time within excerpt (ms)")
path = save_figure(fig, "fig_4_3_waveform_comparison.pdf")
path

PosixPath('/Users/sathira/dev/boss-ds1/ds1-neural-emulation/thesis/images/figures/fig_4_3_waveform_comparison.pdf')

## Figure 4.4 — Representative Spectrogram Comparison

In [8]:
spec_duration = 1.0
spec_series = [("Target", test_target), ("LSTM", outputs["LSTM"]["Best validation"]), ("GRU", outputs["GRU"]["Best validation"])]
specs = []
for label, x in spec_series:
    freqs, times, spec_db = spectrogram_db(x, fs_target, MAIN_START, spec_duration)
    specs.append((label, freqs, times, spec_db))
all_values = np.concatenate([spec_db.ravel() for _, _, _, spec_db in specs])
vmax = np.percentile(all_values, 99)
vmin = vmax - 80
freq_max_khz = min(20.0, fs_target / 2000)

fig, axes = plt.subplots(3, 1, figsize=(7.2, 5.0), sharex=True, sharey=True)
mesh = None
for idx, (ax, (label, freqs, times, spec_db)) in enumerate(zip(axes, specs)):
    mesh = ax.pcolormesh(times - MAIN_START, freqs / 1000, spec_db, shading="auto", cmap="magma", vmin=vmin, vmax=vmax)
    ax.set_ylim(0, freq_max_khz)
    ax.set_ylabel("Frequency (kHz)")
    panel_label(ax, f"({chr(ord('a') + idx)})", x=0.012, y=0.94)
    ax.text(0.10, 0.94, label, transform=ax.transAxes, va="top", ha="left", color="white")
    ax.tick_params(direction="out", width=0.7, length=3)
axes[-1].set_xlabel("Time within excerpt (s)")
cbar = fig.colorbar(mesh, ax=axes, fraction=0.025, pad=0.02)
cbar.set_label("Magnitude (dB)")
path = save_figure(fig, "fig_4_4_spectrogram_comparison.png", dpi=400)
path

PosixPath('/Users/sathira/dev/boss-ds1/ds1-neural-emulation/thesis/images/figures/fig_4_4_spectrogram_comparison.png')

## Figure 4.5 — Accuracy-Efficiency Comparison

In [9]:
plot_rows = []
for name in RUNS:
    params = df_models[df_models["Model"] == name]["Parameters"].iloc[0]
    esr = df_metrics[(df_metrics["Model"] == name) & (df_metrics["Checkpoint"] == "Best validation")]["ESR"].iloc[0]
    plot_rows.append((name, params, esr))

fig, ax = plt.subplots(figsize=(4.6, 3.0))
for name, params, esr in plot_rows:
    ax.scatter(params, esr, s=58, color=RUNS[name]["color"], marker=RUNS[name]["marker"], edgecolor="black", linewidth=0.5, zorder=3)
    offset = (30, -0.00035) if name == "LSTM" else (30, 0.00035)
    ax.text(params + offset[0], esr + offset[1], name, va="center", ha="left")
style_axis(ax)
ax.set_xlabel("Parameter count")
ax.set_ylabel("ESR")
ax.annotate("lower ESR", xy=(0.06, 0.08), xytext=(0.06, 0.23), xycoords="axes fraction", textcoords="axes fraction", arrowprops=dict(arrowstyle="->", linewidth=0.7), ha="center")
path = save_figure(fig, "fig_4_5_accuracy_efficiency.pdf")
path

PosixPath('/Users/sathira/dev/boss-ds1/ds1-neural-emulation/thesis/images/figures/fig_4_5_accuracy_efficiency.pdf')

## Appendix C.1/C.2 — Full-Resolution Loss Curves

In [10]:
appendix_paths = []
for fig_label, name, filename in [
    ("C.1", "LSTM", "appendix_c_1_lstm_loss_curve.pdf"),
    ("C.2", "GRU", "appendix_c_2_gru_loss_curve.pdf"),
]:
    run = RUNS[name]
    s = stats[name]
    fig, ax = plt.subplots(figsize=(6.0, 3.0))
    epochs = np.arange(1, len(s["train_losses"]) + 1)
    ax.plot(epochs, s["train_losses"], color=run["color"], linestyle="-", label="Training")
    val_epochs = [ep for ep, _ in s["val_losses"]]
    val_values = [loss for _, loss in s["val_losses"]]
    ax.plot(val_epochs, val_values, color=run["color"], linestyle="--", marker=run["marker"], markevery=max(len(val_epochs)//14, 1), label="Validation")
    ax.axvline(s["best_epoch"], color="#333333", linestyle=":", linewidth=1.0, label="Best validation")
    style_axis(ax)
    panel_label(ax, f"({fig_label})")
    ax.text(0.13, 0.97, f"{name} loss", transform=ax.transAxes, va="top", ha="left")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.legend(frameon=False, loc="upper right")
    appendix_paths.append(save_figure(fig, filename))
appendix_paths

[PosixPath('/Users/sathira/dev/boss-ds1/ds1-neural-emulation/thesis/images/figures/appendix_c_1_lstm_loss_curve.pdf'),
 PosixPath('/Users/sathira/dev/boss-ds1/ds1-neural-emulation/thesis/images/figures/appendix_c_2_gru_loss_curve.pdf')]

## Appendix — Extra Waveform Examples

In [11]:
extra_starts = EXCERPTS[:3]
wave_duration = 0.12
fig, axes = plt.subplots(len(extra_starts), 3, figsize=(7.2, 5.2), sharex=True, sharey=True)
for row, start_s in enumerate(extra_starts):
    row_segments = []
    for label, x in [("Target", test_target), ("LSTM", outputs["LSTM"]["Best validation"]), ("GRU", outputs["GRU"]["Best validation"] )]:
        t_rel, seg = excerpt_signal(x, fs_target, start_s, wave_duration)
        row_segments.append((label, t_rel * 1000, seg))
    row_ymax = max(np.max(np.abs(seg)) for _, _, seg in row_segments) * 1.05
    for col, (label, t_ms, seg) in enumerate(row_segments):
        ax = axes[row, col]
        color = "#333333" if label == "Target" else RUNS[label]["color"]
        linestyle = "-" if label == "Target" else RUNS[label]["linestyle"]
        ax.plot(t_ms, seg, color=color, linestyle=linestyle, linewidth=0.8)
        style_axis(ax)
        ax.set_ylim(-row_ymax, row_ymax)
        if row == 0:
            ax.text(0.04, 0.92, label, transform=ax.transAxes, va="top", ha="left")
        if col == 0:
            ax.set_ylabel(f"{start_s:.2f} s\nAmplitude")
        if row == len(extra_starts) - 1:
            ax.set_xlabel("Time (ms)")
path = save_figure(fig, "appendix_waveform_examples.pdf")
path

PosixPath('/Users/sathira/dev/boss-ds1/ds1-neural-emulation/thesis/images/figures/appendix_waveform_examples.pdf')

## Appendix — Extra Spectrogram Examples

In [12]:
extra_spec_starts = EXCERPTS[:3]
spec_duration = 1.0
spec_items = []
for start_s in extra_spec_starts:
    for label, x in [("Target", test_target), ("LSTM", outputs["LSTM"]["Best validation"]), ("GRU", outputs["GRU"]["Best validation"] )]:
        freqs, times, spec_db = spectrogram_db(x, fs_target, start_s, spec_duration)
        spec_items.append((start_s, label, freqs, times, spec_db))
all_values = np.concatenate([item[-1].ravel() for item in spec_items])
vmax = np.percentile(all_values, 99)
vmin = vmax - 80
freq_max_khz = min(20.0, fs_target / 2000)

fig, axes = plt.subplots(len(extra_spec_starts), 3, figsize=(7.2, 5.4), sharex=True, sharey=True)
mesh = None
for row, start_s in enumerate(extra_spec_starts):
    for col, label in enumerate(["Target", "LSTM", "GRU"]):
        ax = axes[row, col]
        item = next(v for v in spec_items if v[0] == start_s and v[1] == label)
        _, _, freqs, times, spec_db = item
        mesh = ax.pcolormesh(times - start_s, freqs / 1000, spec_db, shading="auto", cmap="magma", vmin=vmin, vmax=vmax)
        ax.set_ylim(0, freq_max_khz)
        ax.tick_params(direction="out", width=0.7, length=3)
        if row == 0:
            ax.text(0.04, 0.92, label, transform=ax.transAxes, va="top", ha="left", color="white")
        if col == 0:
            ax.set_ylabel(f"{start_s:.2f} s\nFrequency (kHz)")
        if row == len(extra_spec_starts) - 1:
            ax.set_xlabel("Time (s)")
cbar = fig.colorbar(mesh, ax=axes, fraction=0.025, pad=0.02)
cbar.set_label("Magnitude (dB)")
path = save_figure(fig, "appendix_spectrogram_examples.png", dpi=400)
path

PosixPath('/Users/sathira/dev/boss-ds1/ds1-neural-emulation/thesis/images/figures/appendix_spectrogram_examples.png')

## Appendix — Residual Error Waveform

In [13]:
res_duration = 0.12
residuals = []
for name in RUNS:
    _, target_seg = excerpt_signal(test_target, fs_target, MAIN_START, res_duration)
    t_rel, output_seg = excerpt_signal(outputs[name]["Best validation"], fs_target, MAIN_START, res_duration)
    residuals.append((name, t_rel * 1000, output_seg - target_seg))
ymax = max(np.max(np.abs(seg)) for _, _, seg in residuals) * 1.08

fig, axes = plt.subplots(2, 1, figsize=(7.2, 3.2), sharex=True, sharey=True)
for idx, (ax, (name, t_ms, seg)) in enumerate(zip(axes, residuals)):
    ax.axhline(0, color="#333333", linewidth=0.6)
    ax.plot(t_ms, seg, color=RUNS[name]["color"], linestyle=RUNS[name]["linestyle"], linewidth=0.9)
    style_axis(ax)
    panel_label(ax, f"({chr(ord('a') + idx)})")
    ax.text(0.10, 0.92, f"{name} residual", transform=ax.transAxes, va="top", ha="left")
    ax.set_ylabel("Residual")
    ax.set_ylim(-ymax, ymax)
axes[-1].set_xlabel("Time within excerpt (ms)")
path = save_figure(fig, "appendix_residual_waveform.pdf")
path

PosixPath('/Users/sathira/dev/boss-ds1/ds1-neural-emulation/thesis/images/figures/appendix_residual_waveform.pdf')

## Appendix — Spectral Error by Frequency Band

In [14]:
BANDS = [
    ("20-100 Hz", 20, 100),
    ("100-500 Hz", 100, 500),
    ("500 Hz-2 kHz", 500, 2000),
    ("2-5 kHz", 2000, 5000),
    ("5-10 kHz", 5000, 10000),
    ("10-20 kHz", 10000, 20000),
]
rows = []
for name in RUNS:
    for row in spectral_band_errors(test_target, outputs[name]["Best validation"], fs_target, BANDS):
        row["Model"] = name
        rows.append(row)
df_spectral_error = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(7.0, 3.1))
bands = [b[0] for b in BANDS]
x = np.arange(len(bands))
width = 0.34
for offset, name in [(-width / 2, "LSTM"), (width / 2, "GRU")]:
    vals = df_spectral_error[df_spectral_error["Model"] == name].set_index("Band").loc[bands]["Mean Abs Spectral Error (dB)"].values
    ax.bar(x + offset, vals, width=width, label=name, color=RUNS[name]["color"], edgecolor="black", linewidth=0.4, hatch="" if name == "LSTM" else "//")
style_axis(ax)
ax.set_xticks(x)
ax.set_xticklabels(bands, rotation=20, ha="right")
ax.set_ylabel("Mean absolute spectral error (dB)")
ax.set_xlabel("Frequency band")
ax.legend(frameon=False, loc="upper left")
path = save_figure(fig, "appendix_spectral_error.pdf")
path

PosixPath('/Users/sathira/dev/boss-ds1/ds1-neural-emulation/thesis/images/figures/appendix_spectral_error.pdf')

## Appendix — Best-Validation vs Final-Checkpoint ESR

In [15]:
fig, ax = plt.subplots(figsize=(4.8, 3.0))
models = list(RUNS.keys())
x = np.arange(len(models))
width = 0.34
for offset, checkpoint, hatch in [(-width / 2, "Best validation", ""), (width / 2, "Final", "//")]:
    vals = [df_metrics[(df_metrics["Model"] == name) & (df_metrics["Checkpoint"] == checkpoint)]["ESR"].iloc[0] for name in models]
    bars = ax.bar(x + offset, vals, width=width, label=checkpoint, color="#bfbfbf" if checkpoint == "Final" else "#ffffff", edgecolor="#222222", linewidth=0.7, hatch=hatch)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.00045, f"{val:.5f}", ha="center", va="bottom", fontsize=7)
style_axis(ax)
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.set_ylabel("ESR")
ax.set_xlabel("Model")
ax.set_ylim(0, df_metrics["ESR"].max() * 1.20)
ax.legend(frameon=False, loc="upper left")
path = save_figure(fig, "appendix_best_vs_final_esr.pdf")
path

PosixPath('/Users/sathira/dev/boss-ds1/ds1-neural-emulation/thesis/images/figures/appendix_best_vs_final_esr.pdf')

## Export Summary

In [16]:
expected = [
    "fig_4_1_loss_curves.pdf",
    "fig_4_2_learning_rate_schedule.pdf",
    "fig_4_3_waveform_comparison.pdf",
    "fig_4_4_spectrogram_comparison.png",
    "fig_4_5_accuracy_efficiency.pdf",
    "appendix_c_1_lstm_loss_curve.pdf",
    "appendix_c_2_gru_loss_curve.pdf",
    "appendix_waveform_examples.pdf",
    "appendix_spectrogram_examples.png",
    "appendix_residual_waveform.pdf",
    "appendix_spectral_error.pdf",
    "appendix_best_vs_final_esr.pdf",
]
export_rows = []
for filename in expected:
    path = FIGURE_DIR / filename
    export_rows.append({"Figure": filename, "Exists": path.exists(), "Size (KB)": path.stat().st_size / 1024 if path.exists() else 0})
df_exports = pd.DataFrame(export_rows)
display(df_exports.style.format({"Size (KB)": "{:.1f}"}).hide(axis="index"))
assert df_exports["Exists"].all(), "One or more expected figure files were not exported."

Figure,Exists,Size (KB)
fig_4_1_loss_curves.pdf,True,21.2
fig_4_2_learning_rate_schedule.pdf,True,11.2
fig_4_3_waveform_comparison.pdf,True,29.5
fig_4_4_spectrogram_comparison.png,True,474.3
fig_4_5_accuracy_efficiency.pdf,True,11.4
appendix_c_1_lstm_loss_curve.pdf,True,18.8
appendix_c_2_gru_loss_curve.pdf,True,17.3
appendix_waveform_examples.pdf,True,49.8
appendix_spectrogram_examples.png,True,1192.6
appendix_residual_waveform.pdf,True,31.5
